In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260602_112228"

snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots.parquet"))

# RAW DATA
#    ↓
# REGIME FEATURES (no future)
#    ↓
# FIT SCALER + GMM
#    ↓
# ASSIGN REGIMES
#    ↓
# BUILD FUTURE LABELS (separate)
#    ↓
# JOIN ON TIME INDEX
#    ↓
# REGIME STATISTICS / LABELING

snapshots

,ts,symbol,best_bid,best_ask,mid,microprice,best_bid_tick,best_ask_tick,mid_tick,spread,...,ask_delta,quote_churn,future_mid_100ms,future_return_100ms,future_mid_500ms,future_return_500ms,future_mid_1000ms,future_return_1000ms,future_mid_5000ms,future_return_5000ms
0,1780397822814,BTCUSDT,69684.01,69684.02,69684.015,69684.014322,6968401,6968402,6968402,0.01,...,0.0,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0
1,1780397822914,BTCUSDT,69684.01,69684.02,69684.015,69684.014355,6968401,6968402,6968402,0.01,...,0.0,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0
2,1780397823014,BTCUSDT,69684.01,69684.02,69684.015,69684.014355,6968401,6968402,6968402,0.01,...,0.0,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0
3,1780397823114,BTCUSDT,69684.01,69684.02,69684.015,69684.014354,6968401,6968402,6968402,0.01,...,0.0,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0
4,1780397823214,BTCUSDT,69684.01,69684.02,69684.015,69684.014354,6968401,6968402,6968402,0.01,...,0.0,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15243,1780399347114,BTCUSDT,69466.00,69466.01,69466.005,69466.008129,6946600,6946601,6946600,0.01,...,0.0,0.0,69466.005,0.0,69466.005,0.0,69466.005,0.0,69466.005,0.0
15244,1780399347214,BTCUSDT,69466.00,69466.01,69466.005,69466.008129,6946600,6946601,6946600,0.01,...,0.0,0.0,69466.005,0.0,69466.005,0.0,69466.005,0.0,69466.005,0.0
15245,1780399347314,BTCUSDT,69466.00,69466.01,69466.005,69466.008129,6946600,6946601,6946600,0.01,...,0.0,0.0,69466.005,0.0,69466.005,0.0,69466.005,0.0,69466.005,0.0
15246,1780399347414,BTCUSDT,69466.00,69466.01,69466.005,69466.008130,6946600,6946601,6946600,0.01,...,0.0,0.0,69466.005,0.0,69466.005,0.0,69466.005,0.0,69466.005,0.0


In [47]:
# STEP 1 — Load raw data
df = snapshots
df["ts"] = pd.to_datetime(df["ts"], unit="ms")
df = df.set_index("ts")

# STEP 2 — Build REGIME FEATURES (ONLY past info) slower trends - 1000ms

"""
2. Choose regime window (critical design choice)
Start simple:
"""

feature_cols = [
    "volatility",
    "spread",
    "order_imbalance",
    "trade_imbalance",
    "quote_churn",
    "inventory",
    "inventory_vol",
    "microprice_error"
]

WINDOW = "1s"   # later try 2s, 5s

regime_df = pd.DataFrame()

regime_df["volatility"] = df["mid"].pct_change().rolling(WINDOW).std()
regime_df["spread"] = df["spread"].rolling(WINDOW).mean()
regime_df["order_imbalance"] = df["order_imbalance"].rolling(WINDOW).mean()
regime_df["trade_imbalance"] = df["trade_imbalance"].rolling(WINDOW).mean()
regime_df["quote_churn"] = df["quote_churn"].rolling(WINDOW).mean()
regime_df["inventory"] = df["inventory"].rolling(WINDOW).mean()
regime_df["inventory_vol"] = df["inventory"].rolling(WINDOW).std()
regime_df["microprice_error"] = (df["mid"] - df["microprice"]).rolling(WINDOW).mean()

regime_df = regime_df.dropna()

In [48]:
X = regime_df[feature_cols].values
X

array([[ 0.00000000e+00,  1.00000000e-02, -1.28912820e-01, ...,
         0.00000000e+00,  0.00000000e+00,  6.56419812e-04],
       [ 0.00000000e+00,  1.00000000e-02, -1.28370917e-01, ...,
         0.00000000e+00,  0.00000000e+00,  6.53704610e-04],
       [ 0.00000000e+00,  1.00000000e-02, -1.28072948e-01, ...,
         0.00000000e+00,  0.00000000e+00,  6.52211637e-04],
       ...,
       [ 0.00000000e+00,  9.99999999e-03,  6.23146393e-01, ...,
        -7.54390236e-01,  0.00000000e+00, -3.10636913e-03],
       [ 0.00000000e+00,  9.99999999e-03,  6.23830365e-01, ...,
        -7.54390236e-01,  0.00000000e+00, -3.10980467e-03],
       [ 0.00000000e+00,  9.99999999e-03,  6.24514338e-01, ...,
        -7.54390236e-01,  0.00000000e+00, -3.11324021e-03]],
      shape=(15246, 8))

In [41]:
regime_df

,volatility,spread,order_imbalance,trade_imbalance,quote_churn,inventory,inventory_vol,microprice_error
ts,,,,,,,,
2026-06-02 10:57:03.014,0.0,0.01,-0.128913,-0.200000,0.0,0.00000,0.0,0.000656
2026-06-02 10:57:03.114,0.0,0.01,-0.128371,-0.240000,0.0,0.00000,0.0,0.000654
2026-06-02 10:57:03.214,0.0,0.01,-0.128073,-0.264000,0.0,0.00000,0.0,0.000652
2026-06-02 10:57:03.314,0.0,0.01,-0.127864,-0.280000,0.0,0.00000,0.0,0.000651
2026-06-02 10:57:03.414,0.0,0.01,-0.127715,-0.291429,0.0,0.00000,0.0,0.000650
...,...,...,...,...,...,...,...,...
2026-06-02 11:22:27.114,0.0,0.01,0.621826,0.812756,0.0,-0.75439,0.0,-0.003100
2026-06-02 11:22:27.214,0.0,0.01,0.622477,0.821182,0.0,-0.75439,0.0,-0.003103
2026-06-02 11:22:27.314,0.0,0.01,0.623146,0.829608,0.0,-0.75439,0.0,-0.003106


In [49]:
# STEP 3 — Train regime model

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_regimes = 3  # start small: 2–5 max

model = GaussianMixture(
    n_components=n_regimes,
    covariance_type="full",
    random_state=42
)

regime_df["regime"] = model.fit_predict(X_scaled)

In [50]:
# STEP 4 — NOW build evaluation labels (separate dataset)

eval_df = df.copy()
mid = eval_df["mid"].values
h = 10 # 1000ms window

future_return = np.full(len(df), np.nan)  # creates an array the same length as your dataset and fills every element with NaN 
future_volatility = np.full(len(df), np.nan)
future_direction = np.full(len(df), np.nan)

for i in range(len(df) - h):

    p0 = mid[i]
    p1 = mid[i + h]

    window = mid[i:i + h]

    # 1. Return (trend / drift)
    future_return[i] = (p1 - p0) / p0

    # 2. Realized volatility in future window
    future_volatility[i] = np.std(np.diff(window) / window[:-1])

    # 3. Direction (simple sign regime)
    future_direction[i] = np.sign(p1 - p0)

# INSERT HERE (this is the key step)
eval_df["future_return"] = future_return
eval_df["future_volatility"] = future_volatility
eval_df["future_direction"] = future_direction 

# Future direction regime
# If result ≈ +1
# almost always up moves after this regime
# strong bullish bias
# If result ≈ -1
# almost always down moves after this regime
# bearish bias
# If result ≈ 0
# no directional bias
# pure noise / mean reversion / stable

# optional cleanup AFTER labeling
eval_df = eval_df.dropna(subset=["future_return", "future_volatility", "future_direction"])

In [44]:
eval_df

,symbol,best_bid,best_ask,mid,microprice,best_bid_tick,best_ask_tick,mid_tick,spread,order_imbalance,...,future_return_100ms,future_mid_500ms,future_return_500ms,future_mid_1000ms,future_return_1000ms,future_mid_5000ms,future_return_5000ms,future_return,future_volatility,future_direction
ts,,,,,,,,,,,,,,,,,,,,,
2026-06-02 10:57:02.814,BTCUSDT,69684.01,69684.02,69684.015,69684.014322,6968401,6968402,6968402,0.01,-0.133286,...,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,0.0,0.0,0.0
2026-06-02 10:57:02.914,BTCUSDT,69684.01,69684.02,69684.015,69684.014355,6968401,6968402,6968402,0.01,-0.126726,...,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,0.0,0.0,0.0
2026-06-02 10:57:03.014,BTCUSDT,69684.01,69684.02,69684.015,69684.014355,6968401,6968402,6968402,0.01,-0.126726,...,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,0.0,0.0,0.0
2026-06-02 10:57:03.114,BTCUSDT,69684.01,69684.02,69684.015,69684.014354,6968401,6968402,6968402,0.01,-0.126745,...,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,0.0,0.0,0.0
2026-06-02 10:57:03.214,BTCUSDT,69684.01,69684.02,69684.015,69684.014354,6968401,6968402,6968402,0.01,-0.126881,...,0.0,69684.015,0.0,69684.015,0.0,69684.015,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-02 11:22:26.114,BTCUSDT,69466.00,69466.01,69466.005,69466.008096,6946600,6946601,6946600,0.01,0.621124,...,0.0,69466.005,0.0,69466.005,0.0,69466.005,0.0,0.0,0.0,0.0
2026-06-02 11:22:26.214,BTCUSDT,69466.00,69466.01,69466.005,69466.008097,6946600,6946601,6946600,0.01,0.621187,...,0.0,69466.005,0.0,69466.005,0.0,69466.005,0.0,0.0,0.0,0.0
2026-06-02 11:22:26.314,BTCUSDT,69466.00,69466.01,69466.005,69466.008096,6946600,6946601,6946600,0.01,0.620996,...,0.0,69466.005,0.0,69466.005,0.0,69466.005,0.0,0.0,0.0,0.0


In [51]:
# STEP 5 — ALIGN BOTH DATASETS

# This is the missing step in your code.

# Now regime + outcome are aligned.

final = regime_df.merge(
    eval_df[["future_return", "future_volatility", "future_direction"]],
    left_index=True,
    right_index=True,
    how="inner"
)

# STEP 6 — ANALYZE REGIMES

final.groupby("regime").agg({
    "future_return": "mean",
    "future_volatility": "mean",
    "future_direction": "mean"
})

,future_return,future_volatility,future_direction
regime,,,
0,-6.102191e-06,0.000004,-0.087849
1,4.421887e-08,0.000002,-0.006629
2,1.561697e-05,0.000007,0.476190


In [ ]:
feature_cols = [ # What characterizes each regime?
    "volatility",
    "spread",
    "order_imbalance",
    "trade_imbalance",
    "quote_churn",
    "inventory",
    "inventory_vol",
    "microprice_error"
]

z = final.copy()

for col in feature_cols:
    z[col] = (
        z[col] - z[col].mean()
    ) / z[col].std()

regime_profile = (
    z.groupby("regime")[feature_cols]
    .mean()
    .round(2)
)

full_profile = pd.DataFrame(regime_profile.join(regime_outcomes))

"""
🟢 Regime 0
volatility: moderate (0.63)
return: ~0
direction: slightly negative (-0.08)
Meaning:

Efficient / balanced market microstructure

No edge.

This is your baseline state.

🟡 Regime 1
volatility: low (-0.34)
return: ~0
direction: near 0 (-0.006)
Meaning:

Quiet mean-reverting regime

Slight structure but no directional payoff.

Still not tradable.

🔴 Regime 2 (THIS IS THE ONLY IMPORTANT ONE)
volatility: 3.72 (very high)
spread: 23 (extreme)
microprice_error: 10.93 (huge dislocation)
direction: 0.476 (strong bias)
return: positive
Meaning:

This is a liquidity shock / imbalance / breakout regime

This is exactly what HFT desks look for:

order book stress
directional flow imbalance
mispricing between microprice and mid

👉 This is your alpha regime

"""
full_profile

,volatility,spread,order_imbalance,trade_imbalance,quote_churn,inventory,inventory_vol,microprice_error,future_return,future_direction,future_volatility
regime,,,,,,,,,,,
0,0.63,-0.02,-0.22,-0.23,NaN,-0.00,0.32,0.20,-6.102191e-06,-0.087849,0.000004
1,-0.34,-0.04,0.12,0.12,NaN,0.00,-0.17,-0.13,4.421887e-08,-0.006629,0.000002
2,3.72,23.09,-0.84,-0.73,NaN,-0.14,-0.16,10.93,1.561697e-05,0.476190,0.000007


In [52]:
regime_labels = {
    0: "neutral",
    1: "low_vol",
    2: "trending",
}

artifact = {
    "scaler": scaler,
    "model": model,
    "feature_cols": feature_cols,
    "n_regimes": n_regimes,
    "window": WINDOW,
    "regime_labels": regime_labels
}

joblib.dump(artifact, "regime_model_3.pkl")

['regime_model_3.pkl']

In [ ]:
"""
Key Research Questions

This project is designed to investigate:

Which regimes favor passive liquidity provision?

"""